# Selenium 설치

In [5]:
pip install selenium webdriver-manager

Note: you may need to restart the kernel to use updated packages.


# 메인 페이지 내용 크롤링
- 카드 이름

- 카드사

- 캐시백

- 혜택 장소

- 할인

- 해외여부

- 전원실적

# test code

In [8]:
import json
import time
import os
import re
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

def clean_filename(filename):
    """파일명 특수문자 제거"""
    return re.sub(r'[\\/*?:"<>|]', "", filename)

def test_crawl_one_basket():
    # 1. 기존 경로 설정 (상위 폴더의 data/cards)
    save_dir = "../data/cards"
    
    # 폴더가 없을 경우 생성하는 안전 장치
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    
    chrome_options = Options()
    # 테스트이므로 브라우저가 뜨는 것을 확인하기 위해 headless는 끕니다.
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    wait = WebDriverWait(driver, 10)
    
    url = "https://www.card-gorilla.com/card?cate=CHK"
    driver.get(url)

    try:
        # 테스트이므로 '더보기'는 클릭하지 않고 현재 보이는 첫 번째 카드만 잡습니다.
        print("테스트 수집을 시작합니다...")
        time.sleep(3) # 페이지 로딩 대기

        # 2. 카드 리스트 아이템들 가져오기 (이미지의 div.card-container 기준)
        items = driver.find_elements(By.CSS_SELECTOR, "div.card-container")
        
        if not items:
            print("카드를 찾을 수 없습니다. 셀렉터를 확인해주세요.")
            return

        # 3. 첫 번째 카드만 처리 (테스트 로직)
        item = items[0] 
        
        try:
            # --- [ 7가지 항목 한 바구니에 담기 ] ---
            # 이미지 분석을 통한 상대 경로(. 사용) 및 클래스 기반 추출
            
            # 1. 카드 이름 (span.card_name)
            card_name = item.find_element(By.CSS_SELECTOR, "span.card_name").text
            
            # 2. 카드사 (span.card_corp)
            company = item.find_element(By.CSS_SELECTOR, "span.card_corp").text
            
            # 3. 캐시백 (div.sale 내부의 첫 번째 p)
            try:
                cashback = item.find_element(By.CSS_SELECTOR, "div.sale > p:nth-of-type(1)").text
            except:
                cashback = "정보없음"
            
            # 4. 혜택 장소 (div.sale 내부의 두 번째 p)
            try:
                benefit_place = item.find_element(By.CSS_SELECTOR, "div.sale > p:nth-of-type(2)").text
            except:
                benefit_place = "정보없음"
            
            # 5. 할인 (div.sale 내부의 세 번째 p)
            try:
                discount = item.find_element(By.CSS_SELECTOR, "div.sale > p:nth-of-type(3)").text
            except:
                discount = "정보없음"
            
            # 6. 해외여부 (div.ex 내부의 p.in_for)
            try:
                overseas = item.find_element(By.CSS_SELECTOR, "div.ex > p.in_for").text
            except:
                overseas = "정보없음"
            
            # 7. 전월실적 (div.ex 내부의 p.l_mth)
            try:
                performance = item.find_element(By.CSS_SELECTOR, "div.ex > p.l_mth").text
            except:
                performance = "정보없음"

            # 바구니 완성
            card_basket = {
                "card_name": card_name,
                "company": company,
                "cashback": cashback,
                "benefit_place": benefit_place,
                "discount": discount,
                "overseas": overseas,
                "performance": performance
            }
            
            # 4. 파일 저장
            file_name = f"{clean_filename(company)}_{clean_filename(card_name)}.json"
            file_path = os.path.join(save_dir, file_name)

            with open(file_path, 'w', encoding='utf-8') as f:
                json.dump(card_basket, f, ensure_ascii=False, indent=4)
            
            print(f"테스트 저장 완료: {file_path}")
            print("데이터 확인:", card_basket)

        except Exception as e:
            print(f"항목 추출 중 오류 발생: {e}")

    finally:
        driver.quit()
        print("테스트 종료.")

if __name__ == "__main__":
    test_crawl_one_basket()

테스트 수집을 시작합니다...
테스트 저장 완료: ../data/cards\케이뱅크_ONE 체크카드.json
데이터 확인: {'card_name': 'ONE 체크카드', 'company': '케이뱅크', 'cashback': '최대\n1.1% 무제한 캐시백', 'benefit_place': '자주 쓰는 곳에서\n5% 캐시백', 'discount': '3번 결제할 때마다\n1,000원 무제한 캐시백', 'overseas': '해외겸용 없음', 'performance': '전월실적 없음'}
테스트 종료.


# 전체 코드 
- 스크롤 후 `카드 더 보기` 클릭 

In [10]:
import json
import time
import os
import re
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from webdriver_manager.chrome import ChromeDriverManager

def clean_filename(filename):
    """파일명 특수문자 제거"""
    return re.sub(r'[\\/*?:"<>|]', "", filename)

def crawl_cards_progressively():
    # 1. 경로 설정
    save_dir = "../data/cards"
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    
    chrome_options = Options()
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    wait = WebDriverWait(driver, 5)
    
    url = "https://www.card-gorilla.com/card?cate=CHK"
    driver.get(url)
    
    scraped_count = 0

    try:
        while True:
            # --- [ 단계 1: 현재 화면에 보이는 카드들 수집 ] ---
            time.sleep(1) # 로딩 대기
            items = driver.find_elements(By.CSS_SELECTOR, "div.card-container")
            
            for item in items:
                try:
                    # 1. 카드 이름 & 2. 카드사 추출
                    card_name = item.find_element(By.CSS_SELECTOR, "span.card_name").text
                    company = item.find_element(By.CSS_SELECTOR, "span.card_corp").text
                    
                    # 파일명 생성 및 중복 확인 (이미 수집한 카드면 건너뜀)
                    file_name = f"{clean_filename(company)}_{clean_filename(card_name)}.json"
                    file_path = os.path.join(save_dir, file_name)
                    
                    if os.path.exists(file_path):
                        continue # 이미 저장된 파일이 있으면 다음 카드로 이동

                    # 3~5. 혜택 정보 (div.sale > p)
                    benefit_elements = item.find_elements(By.CSS_SELECTOR, "div.sale > p")
                    cashback = benefit_elements[0].text if len(benefit_elements) > 0 else "정보없음"
                    benefit_place = benefit_elements[1].text if len(benefit_elements) > 1 else "정보없음"
                    discount = benefit_elements[2].text if len(benefit_elements) > 2 else "정보없음"
                    
                    # 6. 해외여부 & 7. 전월실적
                    try:
                        overseas = item.find_element(By.CSS_SELECTOR, "div.ex > p.in_for").text
                    except: overseas = "정보없음"
                    
                    try:
                        performance = item.find_element(By.CSS_SELECTOR, "div.ex > p.l_mth").text
                    except: performance = "전월실적 없음"

                    # 데이터 저장
                    card_basket = {
                        "card_name": card_name,
                        "company": company,
                        "cashback": cashback,
                        "benefit_place": benefit_place,
                        "discount": discount,
                        "overseas": overseas,
                        "performance": performance
                    }

                    with open(file_path, 'w', encoding='utf-8') as f:
                        json.dump(card_basket, f, ensure_ascii=False, indent=4)
                    
                    scraped_count += 1
                    print(f"[{scraped_count}] 저장 완료: {file_name}")

                except Exception as e:
                    continue # 개별 카드 오류 시 건너뜀

            # --- [ 단계 2: '더 보기' 버튼 클릭 ] ---
            try:
                # 버튼이 화면에 보일 때까지 스크롤 후 클릭
                more_button = driver.find_element(By.CSS_SELECTOR, "a.lst_more")
                driver.execute_script("arguments[0].scrollIntoView();", more_button)
                time.sleep(0.5)
                driver.execute_script("arguments[0].click();", more_button)
                print("\n--- 더 보기 버튼 클릭 (다음 리스트 로딩) ---")
            except NoSuchElementException:
                print("\n모든 카드를 수집했습니다. (더 보기 버튼 없음)")
                break
            except Exception as e:
                print(f"\n더 보기 클릭 중 오류 발생 또는 종료: {e}")
                break

    finally:
        driver.quit()
        print(f"\n총 {scraped_count}개의 새로운 카드 데이터를 수집했습니다.")

if __name__ == "__main__":
    crawl_cards_progressively()


--- 더 보기 버튼 클릭 (다음 리스트 로딩) ---
[1] 저장 완료: KB국민카드_노리2 체크카드(KB Pay).json
[2] 저장 완료: 신한카드_신한카드 SOL트래블 체크.json
[3] 저장 완료: 네이버페이_네이버페이 머니카드.json
[4] 저장 완료: KB국민카드_KB Youth Club 체크카드.json
[5] 저장 완료: KG모빌리언스_모빌리언스카드.json
[6] 저장 완료: 토스뱅크_토스뱅크 체크카드.json
[7] 저장 완료: KB국민카드_트래블러스 체크카드(토심이).json
[8] 저장 완료: 우체국_개이득 체크카드.json
[9] 저장 완료: KB국민카드_KB 틴업 체크카드.json
[10] 저장 완료: MG새마을금고_더나은 체크카드.json
[11] 저장 완료: 하나카드_네이버페이 머니 하나 체크카드.json
[12] 저장 완료: KB국민카드_나라사랑체크카드.json
[13] 저장 완료: KB국민카드_토심이 첵첵 체크카드.json
[14] 저장 완료: 우리카드_카드의정석 오하CHECK.json
[15] 저장 완료: KB국민카드_노리체크카드.json
[16] 저장 완료: 카카오뱅크_카카오뱅크 프렌즈 체크카드.json
[17] 저장 완료: 신한카드_신한카드 Deep Dream 체크.json
[18] 저장 완료: KB국민카드_노리2 체크카드(Global).json
[19] 저장 완료: 신한카드_신한카드 Hey Young 체크.json

--- 더 보기 버튼 클릭 (다음 리스트 로딩) ---
[20] 저장 완료: 하나카드_트래블로그 체크카드.json
[21] 저장 완료: 신한카드_신한카드 On 체크(잔망루피).json
[22] 저장 완료: KB국민카드_직장인보너스체크카드.json
[23] 저장 완료: KB국민카드_트래블러스 체크카드.json
[24] 저장 완료: 하나카드_달달 하나 체크카드.json
[25] 저장 완료: NH농협카드_NH20해봄체크카드.json
[26] 저장 완료: NH농협카드_K-패스카드(체크).json
[27] 저